# Full-Depth — week 9 results, browsable

Loads the committed result tables (no 13 GB feed or C++ build needed)
and regenerates the headline figure. Full methodology: [docs/analytics-wk9.md](../docs/analytics-wk9.md).

*Scope: single day (2020-01-30), single venue, in-sample; descriptive microstructure.*

In [ ]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

imp = pq.read_table('output/impact_depth.parquet').to_pandas()
reg = pq.read_table('output/ofi_regressions.parquet').to_pandas()

imp.sort_values('AD_shares', ascending=False)[
    ['symbol', 'group', 'AD_shares', 'beta_10s', 'r2_10s', 't_nw_10s']]

In [ ]:
r60 = reg[(reg.horizon_s == 60)].pivot_table(
    index='symbol', columns='design', values='r2')
r60.columns = ['contemporaneous R2', 'one-step-ahead R2']
r60.sort_values('contemporaneous R2', ascending=False)

In [ ]:
import matplotlib.pyplot as plt

x, y = np.log10(imp.AD_shares), np.log10(imp.beta_10s)
slope, icept = np.polyfit(x, y, 1)
fig, ax = plt.subplots(figsize=(8, 6))
for g, c in [('ordinary', 'tab:blue'), ('ETF', 'tab:orange'),
             ('VOL_ETP', 'tab:green'), ('WIDE_TICK', 'tab:purple')]:
    m = imp.group == g
    ax.scatter(imp.AD_shares[m], imp.beta_10s[m], c=c, label=g)
for _, r in imp.iterrows():
    ax.annotate(r.symbol, (r.AD_shares, r.beta_10s), fontsize=7)
xs = np.linspace(x.min(), x.max(), 2)
ax.plot(10**xs, 10**(icept + slope*xs), 'k-',
        label=f'fit: slope {slope:.2f}')
ax.plot(10**xs, 10**(icept + -1*xs), 'k--', alpha=0.4,
        label='CKS reference -1')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('avg touch depth (shares/side)')
ax.set_ylabel('impact coefficient (ticks/share, 10s)')
ax.legend(); ax.set_title('Price impact vs depth — 2020-01-30')
plt.tight_layout()